# 07 Similarity Analysis

## Purpose

This notebook is my manual review space for the glycan embedding results.

The main idea here is:
- choose one saved `best_model/` checkpoint
- start from a few anchor glycans
- compare each anchor against hand-built variants
- look at similarity values, histograms, and relative ordering
- save HTML reports with cartoons so the results are easier to review visually

This is still a manual inference notebook, so it does not automatically pull glycans from the train, validation, or test split.


## Setup note

- code stays in GitHub
- checkpoints and similarity outputs stay in Google Drive
- Colab pulls the repo at the start so the notebook uses the current GitHub version of the helper scripts

Basically: if I update `src/` and push it, this notebook should pick that up the next time I run it cleanly in Colab.


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

# Clone once in a fresh runtime; if the repo is already here, pull the latest
# GitHub changes so notebook reruns do not keep using stale helper code.
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
# Put the repo on the import path so the notebook picks up the local src helpers.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# ==============================================================================
# 1. IMPORT ANALYSIS HELPERS AND DEFINE DRIVE PATHS
# ==============================================================================
from pathlib import Path

from IPython.display import Image, display

from src.similarity import (
    build_variant_model_run_specs,
    export_public_variant_html,
    run_variant_similarity_analysis,
    run_variant_similarity_model_suite,
    validate_variant_similarity_inputs,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
SIMILARITY_RESULTS_DIR = DRIVE_ROOT / 'results' / 'similarity'
SIMILARITY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive root: {DRIVE_ROOT}')
print(f'Checkpoints root: {CHECKPOINTS_DIR}')
print(f'Similarity results root: {SIMILARITY_RESULTS_DIR}')


## Choose the tokenizer family and model suite

Edit the next cell when you want to switch tokenizer families, pretrained experiments, classifier run labels, or the public-export settings.

Supported families in this notebook: `byte_bpe`, `glyberta`, `manual`, `hybrid_char_bpe`, `linkage_block`, `donor_bound`, and `semi_atomic`.

The rest of the notebook stays stable so I can rerun the same anchor-and-variant review across the pretrained MLM checkpoint, the classifier fine-tuned from MLM init, and the random-init classifier without rewriting the qualitative setup each time.


In [ ]:
# ==============================================================================
# 2. CHOOSE THE TOKENIZER FAMILY, PRETRAINED EXPERIMENT, AND MODEL SUITE
# ==============================================================================
# Pick one tokenizer family and the shared pretrained experiment name.
# The suite helper below will build the three standard notebook-7 runs:
# - pretrained MLM checkpoint
# - classifier fine-tuned from MLM initialization
# - classifier fine-tuned from random initialization
SUPPORTED_TOKENIZER_FAMILIES = (
    'byte_bpe',
    'glyberta',
    'manual',
    'hybrid_char_bpe',
    'linkage_block',
    'donor_bound',
    'semi_atomic',
)

TOKENIZER_FAMILY = 'manual'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'
MODEL_SUBDIR = 'best_model'

# Match these to the exact classifier folder names from notebook 10.
CLASSIFIER_MLM_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_mlm'
CLASSIFIER_RANDOM_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_randominit'

# Keep a run label so I can rerun the same model suite without overwriting an
# older manual-review export.
OUTPUT_RUN_LABEL = 'live_extended'
REPORT_TITLE = 'Variant Similarity Report'

# Optional clean browser export, similar to notebook 8. These folders are meant
# for Drive review before I copy the final HTML into the repo's public_reports/.
PUBLIC_EXPORT_ENABLED = True
PUBLIC_EXPORT_PARENT_SUBDIR = 'results/public_reports'
PUBLIC_GITHUB_OWNER = 'hb791-dev'
PUBLIC_GITHUB_REPO = 'glycan-roberta'
PUBLIC_GITHUB_REF = 'main'

if TOKENIZER_FAMILY not in SUPPORTED_TOKENIZER_FAMILIES:
    raise ValueError(f'Unsupported tokenizer family: {TOKENIZER_FAMILY}')

RUN_SPECS = build_variant_model_run_specs(
    checkpoints_dir=CHECKPOINTS_DIR,
    similarity_results_dir=SIMILARITY_RESULTS_DIR,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    output_run_label=OUTPUT_RUN_LABEL,
    report_title=REPORT_TITLE,
    model_subdir=MODEL_SUBDIR,
    classifier_mlm_run_label=CLASSIFIER_MLM_RUN_LABEL,
    classifier_random_run_label=CLASSIFIER_RANDOM_RUN_LABEL,
)

PUBLIC_EXPORT_PARENT_DIR = DRIVE_ROOT / PUBLIC_EXPORT_PARENT_SUBDIR
PUBLIC_EXPORT_PARENT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Tokenizer family: {TOKENIZER_FAMILY}')
print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Output run label: {OUTPUT_RUN_LABEL}')
print(f'Public export enabled: {PUBLIC_EXPORT_ENABLED}')
print('Planned model suite:')
for spec in RUN_SPECS:
    print(f"- {spec['model_label']}")
    print(f"  model_dir: {spec['model_dir']}")
    print(f"  output_dir: {spec['output_dir']}")
    print(f"  public_reports repo subdir: {spec['public_report_subdir']}")


## Choose anchor glycans and manual variant sets

This is the main content cell to edit after `MODEL_DIR`.
Keep the variants grouped under each anchor so it is easy to review how every change relates back to the starting glycan.

I like this layout better than a giant flat list because I can sanity-check each little family of edits before I run the notebook.


In [ ]:
# ==============================================================================
# 3. CONFIGURE ANCHORS AND MANUAL VARIANT SETS
# ==============================================================================
# Keep edits grouped by anchor glycan so the review set is easy to inspect.
ANCHOR_GROUPS = [
    {
        'anchor_id': 'A1',
        'anchor_sequence': 'Galb1-3GalNAca',
        'variant_sets': {
            'linkage': [
                {
                    'variant_id': 'A1-L1',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'Galb1-3 -> Galb1-4',
                    'variant_sequence': 'Galb1-4GalNAca',
                },
                {
                    'variant_id': 'A1-L2',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'Galb1-3 -> Gala1-3',
                    'variant_sequence': 'Gala1-3GalNAca',
                },
                {
                    'variant_id': 'A1-L3',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'Galb1-3 -> Gal?1-3',
                    'variant_sequence': 'Gal?1-3GalNAca',
                },
            ],
            'monosaccharide': [
                {
                    'variant_id': 'A1-M1',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'Gal -> Glc',
                    'variant_sequence': 'Glcb1-3GalNAca',
                },
                {
                    'variant_id': 'A1-M2',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'GalNAc -> GlcNAc',
                    'variant_sequence': 'Galb1-3GlcNAca',
                },
                {
                    'variant_id': 'A1-M3',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'Gal -> Fuc',
                    'variant_sequence': 'Fucb1-3GalNAca',
                },
            ],
            'branch_terminal': [
                {
                    'variant_id': 'A1-B1',
                    'edit_type': 'branch_addition',
                    'edit_description': 'Add branch GlcNAca1-4',
                    'variant_sequence': 'Galb1-3(GlcNAca1-4)GalNAca',
                },
                {
                    'variant_id': 'A1-B2',
                    'edit_type': 'terminal_extension',
                    'edit_description': 'Add terminal a1-3Gal',
                    'variant_sequence': 'Galb1-3GalNAca1-3Gal',
                },
                {
                    'variant_id': 'A1-B3',
                    'edit_type': 'branch_addition',
                    'edit_description': 'Add branch Fuca1-2',
                    'variant_sequence': 'Galb1-3(Fuca1-2)GalNAca',
                },
            ],
        },
    },
    {
        'anchor_id': 'A2',
        'anchor_sequence': 'Mana1-3(Mana1-6)Manb1-4GlcNAcb',
        'variant_sets': {
            'linkage': [
                {
                    'variant_id': 'A2-L1',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'Mana1-3 -> Mana1-4',
                    'variant_sequence': 'Mana1-4(Mana1-6)Manb1-4GlcNAcb',
                },
                {
                    'variant_id': 'A2-L2',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'Mana1-6 branch -> Mana1-4 branch',
                    'variant_sequence': 'Mana1-3(Mana1-4)Manb1-4GlcNAcb',
                },
                {
                    'variant_id': 'A2-L3',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'Manb1-4 -> Mana1-4',
                    'variant_sequence': 'Mana1-3(Mana1-6)Mana1-4GlcNAcb',
                },
            ],
            'monosaccharide': [
                {
                    'variant_id': 'A2-M1',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'Outer Man -> Gal',
                    'variant_sequence': 'Gala1-3(Mana1-6)Manb1-4GlcNAcb',
                },
                {
                    'variant_id': 'A2-M2',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'Core Man -> Gal',
                    'variant_sequence': 'Mana1-3(Mana1-6)Galb1-4GlcNAcb',
                },
                {
                    'variant_id': 'A2-M3',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'GlcNAc -> GalNAc',
                    'variant_sequence': 'Mana1-3(Mana1-6)Manb1-4GalNAcb',
                },
            ],
            'branch_terminal': [
                {
                    'variant_id': 'A2-B1',
                    'edit_type': 'branch_addition',
                    'edit_description': 'Add branch Gala1-3',
                    'variant_sequence': 'Mana1-3(Mana1-6)(Gala1-3)Manb1-4GlcNAcb',
                },
                {
                    'variant_id': 'A2-B2',
                    'edit_type': 'terminal_extension',
                    'edit_description': 'Add terminal b1-4Gal',
                    'variant_sequence': 'Mana1-3(Mana1-6)Manb1-4GlcNAcb1-4Gal',
                },
                {
                    'variant_id': 'A2-B3',
                    'edit_type': 'branch_addition',
                    'edit_description': 'Add branch Fuca1-6',
                    'variant_sequence': 'Mana1-3(Mana1-6)Manb1-4(Fuca1-6)GlcNAcb',
                },
            ],
        },
    },
    {
        'anchor_id': 'A3',
        'anchor_sequence': 'Galb1-3GlcNAc?1-3(NeuAca2-6)GalNAca',
        'variant_sets': {
            'linkage': [
                {
                    'variant_id': 'A3-L1',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'Galb1-3 -> Galb1-4',
                    'variant_sequence': 'Galb1-4GlcNAc?1-3(NeuAca2-6)GalNAca',
                },
                {
                    'variant_id': 'A3-L2',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'GlcNAc?1-3 -> GlcNAc?1-6',
                    'variant_sequence': 'Galb1-3GlcNAc?1-6(NeuAca2-6)GalNAca',
                },
                {
                    'variant_id': 'A3-L3',
                    'edit_type': 'linkage_swap',
                    'edit_description': 'NeuAca2-6 -> NeuAca2-3',
                    'variant_sequence': 'Galb1-3GlcNAc?1-3(NeuAca2-3)GalNAca',
                },
            ],
            'monosaccharide': [
                {
                    'variant_id': 'A3-M1',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'Gal -> Glc',
                    'variant_sequence': 'Glcb1-3GlcNAc?1-3(NeuAca2-6)GalNAca',
                },
                {
                    'variant_id': 'A3-M2',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'GlcNAc -> GalNAc',
                    'variant_sequence': 'Galb1-3GalNAc?1-3(NeuAca2-6)GalNAca',
                },
                {
                    'variant_id': 'A3-M3',
                    'edit_type': 'monosaccharide_swap',
                    'edit_description': 'NeuAc -> NeuGc',
                    'variant_sequence': 'Galb1-3GlcNAc?1-3(NeuGca2-6)GalNAca',
                },
            ],
            'branch_terminal': [
                {
                    'variant_id': 'A3-B1',
                    'edit_type': 'branch_addition',
                    'edit_description': 'Add branch Fuca1-4',
                    'variant_sequence': 'Galb1-3GlcNAc?1-3(NeuAca2-6)(Fuca1-4)GalNAca',
                },
                {
                    'variant_id': 'A3-B2',
                    'edit_type': 'terminal_extension',
                    'edit_description': 'Add terminal a1-3Gal',
                    'variant_sequence': 'Galb1-3GlcNAc?1-3(NeuAca2-6)GalNAca1-3Gal',
                },
                {
                    'variant_id': 'A3-B3',
                    'edit_type': 'terminal_extension',
                    'edit_description': 'Add terminal a1-6GlcNAc',
                    'variant_sequence': 'Galb1-3GlcNAc?1-3(NeuAca2-6)GalNAca1-6GlcNAc',
                },
            ],
        },
    },
]

# Flatten the grouped review sets into the record format expected by the helper.
VARIANT_RECORDS = []
for anchor_group in ANCHOR_GROUPS:
    for variant_set, variants in anchor_group['variant_sets'].items():
        for variant in variants:
            VARIANT_RECORDS.append(
                {
                    'anchor_id': anchor_group['anchor_id'],
                    'anchor_sequence': anchor_group['anchor_sequence'],
                    'variant_set': variant_set,
                    'variant_id': variant['variant_id'],
                    'edit_type': variant['edit_type'],
                    'edit_description': variant['edit_description'],
                    'variant_sequence': variant['variant_sequence'],
                }
            )

# Cartoon lookup is best-effort, but having the images makes the HTML reports much easier to scan.
CARTOON_DEVELOPER_EMAIL = 'PUT EMAIL IN'
CARTOON_IMAGE_FORMAT = 'svg'
CARTOON_DISPLAY = 'compact'
LOOKUP_TIMEOUT = 60

# Leave MAX_LENGTH as None unless I specifically need to test truncation behavior.
MAX_LENGTH = None
BATCH_SIZE = 32

print(f'Anchor groups configured: {len(ANCHOR_GROUPS)}')
print(f'Variant records configured: {len(VARIANT_RECORDS)}')
for anchor_group in ANCHOR_GROUPS:
    anchor_variant_count = sum(len(variants) for variants in anchor_group['variant_sets'].values())
    print(f"- {anchor_group['anchor_id']}: {anchor_variant_count} variants")


## What I expect from the outputs

The main things I want to check are:
- do the similarity scores spread out differently across edit types?
- do the ranked variants look intuitive within each anchor?
- do the histograms make some anchors look much more stable or fragile than others?
- do the cartoons make it easier to explain the results to someone else later?

If the tables look okay but the ordering plots look weird, that is still useful because it probably means the embeddings are not respecting the edits the way I expected.


In [ ]:
# ==============================================================================
# 4. RUN THE ANALYSIS FOR EACH MODEL, DISPLAY THE TABLES, AND SAVE THE OUTPUTS
# ==============================================================================
# Catch malformed variant records before I load any models.
if not RUN_SPECS:
    raise ValueError('RUN_SPECS is empty. Set at least one model in the suite cell above.')

for spec in RUN_SPECS:
    validate_variant_similarity_inputs(
        model_dir=spec['model_dir'],
        variant_records=VARIANT_RECORDS,
        output_dir=spec['output_dir'],
    )

# One helper call now handles the sequential load -> embed -> save cycle for the
# pretrained MLM, the MLM-initialized classifier, and the random-init classifier.
suite_results = run_variant_similarity_model_suite(
    model_specs=RUN_SPECS,
    variant_records=VARIANT_RECORDS,
    developer_email=CARTOON_DEVELOPER_EMAIL,
    cartoon_image_format=CARTOON_IMAGE_FORMAT,
    cartoon_display=CARTOON_DISPLAY,
    lookup_timeout=LOOKUP_TIMEOUT,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
)

public_export_results = {}

SUMMARY_SCOPE_ORDER = {'all_variants': 0, 'linkage': 1, 'monosaccharide': 2, 'branch_terminal': 3}

for spec in RUN_SPECS:
    model_id = spec['model_id']
    model_label = spec['model_label']
    results = suite_results[model_id]['results_bundle']
    variant_results_df = results['variant_results_df']

    print('\n' + '=' * 88)
    print(f'=== {model_label} ===')
    print(f"Model directory: {spec['model_dir']}")
    print(f"Output directory: {spec['output_dir']}")

    # Build the display summary directly from the variant-level results so each
    # anchor definitely shows one all-9 row plus one row for each edit family.
    overall_distribution_summary_df = (
        variant_results_df.groupby('anchor_id', sort=False)['cosine_similarity']
        .agg(['count', 'mean', 'median', 'min', 'max', 'std'])
        .reset_index()
        .rename(columns={'std': 'std_dev'})
    )
    overall_distribution_summary_df['std_dev'] = overall_distribution_summary_df['std_dev'].fillna(0.0)
    overall_distribution_summary_df.insert(1, 'summary_scope', 'all_variants')

    set_distribution_summary_df = (
        variant_results_df.groupby(['anchor_id', 'variant_set'], sort=False)['cosine_similarity']
        .agg(['count', 'mean', 'median', 'min', 'max', 'std'])
        .reset_index()
        .rename(columns={'variant_set': 'summary_scope', 'std': 'std_dev'})
    )
    set_distribution_summary_df['std_dev'] = set_distribution_summary_df['std_dev'].fillna(0.0)

    display_distribution_summary_df = overall_distribution_summary_df._append(
        set_distribution_summary_df,
        ignore_index=True,
    )

    for anchor_id, anchor_df in variant_results_df.groupby('anchor_id', sort=False):
        print(f'=== {anchor_id} anchor-to-variant similarities ===')
        display(
            anchor_df.sort_values(['rank_within_anchor', 'variant_id'], kind='stable')[
                [
                    'rank_within_anchor',
                    'variant_set',
                    'variant_id',
                    'edit_type',
                    'edit_description',
                    'cosine_similarity',
                    'variant_sequence',
                ]
            ].rename(
                columns={
                    'rank_within_anchor': 'overall_anchor_rank',
                }
            )
        )

    print('=== Tokenization preview ===')
    display(results['tokenization_preview_df'])

    print('=== Distribution summary ===')
    for anchor_id, distribution_df in display_distribution_summary_df.groupby('anchor_id', sort=False):
        print(f'--- {anchor_id} distribution summary ---')
        ordered_distribution_df = distribution_df.assign(
            _summary_scope_order=distribution_df['summary_scope'].map(SUMMARY_SCOPE_ORDER)
        ).sort_values(['_summary_scope_order', 'summary_scope'], kind='stable').drop(columns=['_summary_scope_order'])
        display(ordered_distribution_df)

    print('=== Cartoon manifest ===')
    display(results['cartoon_manifest_df'])

    print('=== Overall similarity histogram ===')
    display(Image(filename=str(results['saved_paths']['overall_histogram_path'])))

    for anchor_id in variant_results_df['anchor_id'].drop_duplicates():
        print(f'=== {anchor_id} similarity histogram ===')
        display(Image(filename=str(results['saved_paths']['anchor_histogram_paths'][anchor_id])))

    if PUBLIC_EXPORT_ENABLED:
        export_dir = PUBLIC_EXPORT_PARENT_DIR / spec['public_export_subdir']
        public_export_results[model_id] = export_public_variant_html(
            results_bundle=results,
            export_dir=export_dir,
            repo_public_subdir=spec['public_report_subdir'],
            repo_owner=PUBLIC_GITHUB_OWNER,
            repo_name=PUBLIC_GITHUB_REPO,
            repo_ref=PUBLIC_GITHUB_REF,
            extra_blocked_strings=[value for value in [CARTOON_DEVELOPER_EMAIL] if str(value).strip()],
        )
        print('=== Public export ===')
        print(f"Drive export folder: {public_export_results[model_id]['public_export_dir']}")
        print(f"Repo public_reports target: {spec['public_report_subdir']}")
        print(f"Planned GitHack URL after copy/push: {public_export_results[model_id]['githack_url']}")
        if public_export_results[model_id]['has_sensitive_matches']:
            print('Sensitive-string scan matches found:')
            display(public_export_results[model_id]['scan_results_df'])
        if public_export_results[model_id]['has_dependency_issues']:
            print('Dependency issues found in export:')
            display(public_export_results[model_id]['dependency_issues_df'])

    print('Saved outputs:')
    for label, path in results['saved_paths'].items():
        print(f'- {label}: {path}')
